# Stage 1: Data Sourcing

**Financial Forecasting & Variance Analysis** | Stage 1 of 6 | [Project README](../../README.md) | [Stage 2: Data Ingestion](../02-data-ingestion/02-data-ingestion.ipynb) →

> Define the research question, identify the source data and confirm that every source file is available in OneDrive before anything is loaded.

| Reads | Writes |
| --- | --- |
| OneDrive `TeslimDataLakes/financial_transactions_dataset_analytics_kaggle/` (file listing only) | Nothing. This stage only documents the source and checks it is available |

## 1.1 Research question

> **How predictable is a card portfolio's monthly spending, which customers deviate from their expected spending, and where are value and fraud concentrated?**

The question has three parts, and each one maps to a section of stage 4:

| # | Sub-question | Method | Measure of success |
| --- | --- | --- | --- |
| 1 | Can next month's portfolio **net spend** be forecast more accurately than simple benchmarks? | Walk-forward backtest comparing naive, seasonal naive, SARIMA and XGBoost | MAPE and RMSE against the seasonal-naive benchmark |
| 2 | Which customers and months **deviate materially** from their expected (budgeted) spending, and is the deviation systematic? | Variance against a trailing three-month budget, with significance tests at customer level | Share of material variances; sign and significance of the mean variance |
| 3 | How **concentrated** is customer value, and where is labelled **fraud** concentrated? | RFM segmentation, K-means cross-check, Pareto analysis, fraud-rate profiling | Segment value shares; fraud rate by attribute |

**Scope note.** The data records cardholder transactions. It has no issuer revenue (interchange, fees or interest), so the project forecasts **transaction value (net spend)**, not revenue. Earlier versions of this project called the measure "revenue", which was inaccurate.

## 1.2 Source data

**Publisher:** [Financial Transactions Dataset: Analytics](https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets) on Kaggle (computingvictor).
**Storage used by this project:** OneDrive, `TeslimDataLakes/financial_transactions_dataset_analytics_kaggle/`, read with the [`onedrive_loader`](https://github.com/UthmanAdeyanju/OneDrive-Connector) package through the Microsoft Graph API. The data is never re-downloaded from Kaggle.

| File | Grain (one row per…) | Key | Joins to | Contents |
| --- | --- | --- | --- | --- |
| `transactions_data.csv` | card transaction | `id` → `transaction_id` | `client_id` → users, `card_id` → cards, `mcc` → MCC codes | Timestamp, amount, entry method, merchant, MCC, error codes |
| `users_data.csv` | customer | `id` → `user_id` | (none) | Age, gender, location, income, debt, credit score |
| `cards_data.csv` | card | `id` → `card_id` | `client_id` → users | Brand, type, credit limit, chip, open/expiry dates, dark-web flag |
| `mcc_codes.json` | merchant category | MCC code | (none) | `{code: description}` dictionary |
| `train_fraud_labels.json` | labelled transaction | transaction id | transactions | `{"target": {id: "Yes"/"No"}}`, covering **a subset** of transactions |

```
users (user_id) ──< cards (card_id, user_id)
   │                    │
   └──────< transactions (transaction_id, user_id, card_id, mcc_code) >── mcc_codes (mcc_code)
                         │
                         └── fraud_labels (transaction_id)   [partial coverage]
```

In [1]:
import sys
from pathlib import Path

# Locate the repo root (the folder that holds python/config.py) and import the shared modules
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "python" / "config.py").exists())
sys.path.insert(0, str(ROOT / "python"))

import pandas as pd
import config

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1.3 Source availability check

List the OneDrive folder and confirm that every file the pipeline needs is there. Nothing is downloaded at this stage.

In [2]:
from onedrive_loader import files

available = files(config.ONEDRIVE_FOLDER)

inventory = pd.DataFrame(
    [(table, filename) for table, (filename, _) in config.SOURCES.items()],
    columns=["table", "source_file"],
)
inventory["on_onedrive"] = inventory["source_file"].isin(available)
inventory["expected_rows"] = inventory["table"].map(config.EXPECTED_ROWS)
inventory

,table,source_file,on_onedrive,expected_rows
0,users,users_data.csv,True,2000
1,cards,cards_data.csv,True,6146
2,mcc_codes,mcc_codes.json,True,109
3,fraud_labels,train_fraud_labels.json,True,8914963
4,transactions,transactions_data.csv,True,13305915


In [3]:
missing = inventory.loc[~inventory["on_onedrive"], "source_file"].tolist()
extra = sorted(set(available) - set(inventory["source_file"]))

if extra:
    print("Files in the folder that the pipeline does not use:", extra)
if missing:
    raise FileNotFoundError(f"Upload these files to OneDrive before running stage 2: {missing}")
print(f"All {len(inventory)} source files are available in OneDrive.")

All 5 source files are available in OneDrive.


## 1.4 Source controls and known limitations

| Control | How it is applied |
| --- | --- |
| Single source of truth | Every stage reads from the OneDrive folder (stage 2) or from the local cache built from it. No hand-edited copies |
| Row-count reconciliation | Stage 2 compares the rows loaded with the counts published on Kaggle (`config.EXPECTED_ROWS`) |
| Raw data kept unchanged | Bronze keeps every value as text, exactly as published. Typing happens only in stage 3 |
| Sensitive fields | `card_number` and `cvv` are dropped at silver and never reach analysis or reporting outputs |

**Limitations to carry through the project:**

1. **Provenance.** The Kaggle page doesn't document how the records were produced. Read the findings as a demonstration of method, not as statements about a real bank's portfolio.
2. **Partial fraud labels.** Only part of the transactions have a label. Fraud rates are calculated on labelled transactions only, and unlabelled transactions are never assumed to be genuine.
3. **No revenue or budget data.** "Budget" in the variance analysis is a statistical expectation (a trailing average), not an approved financial budget.
4. **Edge months.** If the extract starts or ends part-way through a month, that month is flagged and excluded from forecasting and variance analysis (stage 4).

Next: [Stage 2: Data Ingestion](../02-data-ingestion/02-data-ingestion.ipynb)